<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM3510-BDA1/main/opim3510_banner.svg" width="100%" alt="OPIM 3510 banner"/>

<a href="https://colab.research.google.com/github/drdave-teaching/OPIM3510-BDA1/blob/main/M2_Forecasting/Week09_Project2/Project2_Forecasting_CT_Demand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2: Forecast Connecticut's Electricity Demand
**OPIM 3510 · Module 2 · Team project · Dr. Dave Wanik**

-----------------------------------------
Every morning, a grid operator forecasts **tomorrow's total electricity demand** for Connecticut. Your job: **beat the simple forecast.** Train on 2011–2019, test on **2020**, forecast 1 day ahead.

📋 Read the [project brief, rules and rubric](https://github.com/drdave-teaching/OPIM3510-BDA1/tree/main/M2_Forecasting/Week09_Project2) first.

Cells marked with comments only are yours to fill in. The Week 6–8 notebooks have everything you need.

# 🔴 0. Setup
Run these cells as-is. They load the data, our `evaluate()` scorecard, and a helper from Week 8.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing

DATA_URL = "https://raw.githubusercontent.com/drdave-teaching/OPIM3510-BDA1/main/data/ct_daily_demand_2011_2021.csv"
daily = pd.read_csv(DATA_URL, index_col='Datetime', parse_dates=True)
daily = daily.asfreq('D') # evenly spaced days
daily.head()

In [ ]:
def evaluate(actual, predicted, naive=None):
    """Score a forecast. Returns a pandas Series you can stack into a leaderboard.

    actual    : the true values (Series, list, or numpy array)
    predicted : your forecast for the same time steps
    naive     : (optional) a baseline forecast for the same time steps.
                If you give one, you also get "skill vs naive (%)":
                how much smaller your MAE is than the baseline's MAE.
                Positive = you beat the baseline. Negative = the baseline wins.

    Rows where any of the forecasts is missing (NaN) are dropped first,
    so every model is scored on exactly the same time steps.
    """
    # put everything side by side so the rows line up
    scores = pd.DataFrame({"actual": actual, "predicted": predicted})
    if naive is not None:
        scores["naive"] = naive

    # only score time steps where we have every number
    scores = scores.dropna()

    # error = predicted - actual
    error = scores["predicted"] - scores["actual"]

    # the four forecast KPIs
    bias = error.mean()
    mae = error.abs().mean()
    rmse = np.sqrt((error ** 2).mean())
    mape = (error.abs() / scores["actual"].abs()).mean() * 100

    results = {
        "n": len(scores),
        "bias": bias,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
    }

    # how much better than the baseline?
    if naive is not None:
        naive_error = scores["naive"] - scores["actual"]
        naive_mae = naive_error.abs().mean()
        results["skill vs naive (%)"] = (1 - mae / naive_mae) * 100

    return pd.Series(results).round(2)

In [ ]:
def holt_winters_one_step(series, fitted, seasonal_periods):
    # Run a Holt-Winters model forward over the whole series using the settings
    # learned on the training data (fitted.params). Nothing is re-learned. (From Week 8.)
    params = fitted.params
    locked = ExponentialSmoothing(series,
                                  trend=None,
                                  seasonal='add',
                                  seasonal_periods=seasonal_periods,
                                  initialization_method='known',
                                  initial_level=params['initial_level'],
                                  initial_seasonal=params['initial_seasons'])
    locked_fit = locked.fit(smoothing_level=params['smoothing_level'],
                            smoothing_seasonal=params['smoothing_seasonal'],
                            optimized=False)
    return locked_fit.fittedvalues

# 🔴 1. Split by time
**Train: 2011–2019. Test: 2020.** Every choice you make uses `train` only.

In [ ]:
# slice the training and test years
# check your work: dates and number of days in each

# your code here

# 🔴 Part A. Honest baselines (25 pts)
## A1. Naive (done for you - use it as a pattern)
Tomorrow = today. `shift(1)` lines each day up with yesterday's demand.

In [ ]:
daily['Naive'] = daily['Demand_GWh'].shift(1)
daily[['Demand_GWh', 'Naive']].head()

## A2. Weekday-adjusted naive
Yesterday's demand + (typical demand for today's weekday − typical demand for yesterday's weekday). **Compute the weekday averages on `train` only.** (Week 7 notebook, section "Can we beat naive?")

In [ ]:
# average demand for each day of the week - TRAINING YEARS ONLY
# today's weekday and yesterday's weekday
# the adjustment and the forecast

# your code here

## A3. Moving average: choose the window on train
Try several windows, score each on **train**, keep the best, then build that forecast.

In [ ]:
# try several windows on the training years
# keep the best window and build that forecast

# your code here

## A4. Holt-Winters (weekly season): fit on train, run forward through 2020

In [ ]:
# fit on the training years only
# lock the settings and make 1-day-ahead forecasts for every day

# your code here

## A5. Leaderboard on the 2020 test year
Score every model with `evaluate()` (pass `naive=` so you get skill vs. naive) and stack the results, best MAE on top.

In [ ]:
# score each model on 2020

# your code here

# 🔴 Part B. Beat it (25 pts)
Pick **at least one** idea. Make every choice on **train** only, then add your model to the leaderboard.

**Idea 1: Weather.** Demand jumps when it gets hot (AC) and climbs when it gets cold (heat, lights). If tomorrow will be hotter than today, the weekday-adjusted naive will be too low. *Degree days* measure how far the temperature is from a comfortable 65°F:
* **Cooling degree days (CDD)** = how many degrees *above* 65°F
* **Heating degree days (HDD)** = how many degrees *below* 65°F

Use the **change** in CDD and HDD from yesterday to today to adjust your forecast. ⚠️ This uses today's actual temperature, so you must say you're **assuming a perfect weather forecast** (Rule 3).

**Idea 2: Holidays.** On a holiday, demand looks like a weekend. The day *after* a holiday, the forecast starts from a quiet day. Measure both effects on train, then add them.

**Idea 3: Blend.** Average two forecasts that make different mistakes.

In [ ]:
# Idea 1: weather. Degree days from the average temperature
# how much did they change from yesterday to today?
# what our best baseline gets wrong (actual - forecast)

# your code here

In [ ]:
# learn how many GWh each degree of change is worth - TRAINING YEARS ONLY
# (a linear regression is an Excel trendline with more than one X)

# your code here

In [ ]:
# adjust the weekday-adjusted naive for the change in weather (assuming a perfect weather forecast)

# your code here

In [ ]:
# Idea 2: holidays. Mark US federal holidays and the day after
# how far off is the weather model on those days? - TRAINING YEARS ONLY

# your code here

In [ ]:
# add the holiday effects to the forecast

# your code here

In [ ]:
# Idea 3: blend - average Holt-Winters and the weather model

# your code here

In [ ]:
# leaderboard v2 on 2020

# your code here

# 🔴 Part C. Diagnose your best model (15 pts)
Plot its errors over 2020, list the 10 worst days, and explain them.

In [ ]:
# errors of the best model (error = predicted - actual)

# your code here

In [ ]:
# the 10 worst days

# your code here

In [ ]:
# average error by month - did the COVID shutdown show up?

# your code here

# 🔴 Part D. Business memo (10 pts)
Write 5 bullets for a manager who doesn't code: **which model**, **how accurate** (in GWh *and* %), **how much better than naive**, **when it fails**, and **what you'd do next**.

*(Your memo here.)*

# ✅ Before you submit
- [ ] Notebook runs top to bottom (**Runtime → Restart and run all**)
- [ ] Every forecast uses `shift(1)` or a training-only setting - no peeking
- [ ] If you used temperature: you wrote *"assuming a perfect weather forecast"*
- [ ] `AI_LOG.md` filled in, with how you **checked** the AI
- [ ] ≥5 meaningful commits, spread over the project
- [ ] Team `README.md` with each member's video link